In [1]:
!curl -I https://google.com 2>&1 | head -5

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0   220    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0




In [2]:
!pip install -q transformers accelerate bitsandbytes datasets scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 81.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 2

In [3]:
from kaggle_secrets import UserSecretsClient
import os

secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
print("Token loaded:", os.environ["HF_TOKEN"][:8], "...")

Token loaded: hf_biSNj ...


In [4]:
%%writefile phase1_baselines.py
"""
phase1_baselines.py
===================
Fragility Under Constraint — Phase 1: Baseline Evaluation
Target hardware: Kaggle T4 (15 GB VRAM)

What this script does
---------------------
Runs all 5 models × 3 tiers × 3 domains on CLEAN data (no shift).
Collects per-cell: Accuracy, F1, ECE proxy, inference latency.
Writes results to baseline_results.csv and baseline_results.json.

Models (5)
----------
Gemma 3 4B, Mistral 7B, Llama 3.1 8B, Qwen 2.5 3B, SmolLM 1.7B

Tiers (3)
---------
Tier 0: FP16  |  Tier 1: INT8  |  Tier 2: INT4

Domains (3) × Tasks (1 per domain for baseline)
------------------------------------------------
Domain A — General NLP       : SST-2 sentiment (binary classification)
Domain B — Financial NLP     : Twitter Financial Sentiment (3-class)
Domain C — Structured Reason : SVAMP math word problems (exact match)

Samples per cell: 50  (fast but statistically meaningful for baseline)
Total cells     : 5 × 3 × 3 = 45
Estimated time  : 3–5 hours on T4

Usage
-----
# Full run
python phase1_baselines.py --hf-token YOUR_TOKEN

# Single model (for testing)
python phase1_baselines.py --model gemma3 --hf-token YOUR_TOKEN

# Single model + single tier
python phase1_baselines.py --model gemma3 --tiers fp16 --hf-token YOUR_TOKEN

# Resume from checkpoint (skips already-completed cells)
python phase1_baselines.py --resume --hf-token YOUR_TOKEN
"""

import argparse
import csv
import gc
import json
import logging
import os
import re
import sys
import time
import traceback
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

# ---------------------------------------------------------------------------
# Logging
# ---------------------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
)
log = logging.getLogger("phase1")

# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------
SAMPLES_PER_CELL = 50       # samples per model × tier × domain cell
VRAM_LIMIT_GB    = 14.0
CHECKPOINT_FILE  = "phase1_checkpoint.json"
RESULTS_CSV      = "baseline_results.csv"
RESULTS_JSON     = "baseline_results.json"

# Models needing eager attention (no flash-attn on Kaggle)
EAGER_ATTN_MODELS = {"qwen25"}

# Models that must use bfloat16 (float16 causes NaN logits)
BFLOAT16_MODELS = {"gemma3"}

# Models that use chat template for generation
# All instruct-tuned models should use this
CHAT_TEMPLATE_MODELS = {"gemma3", "mistral7b", "llama31", "qwen25", "smollm"}

# Per-model eos token ids (in addition to tokenizer default)
EXTRA_EOS_TOKENS = {
    "gemma3": [1, 107],   # <eos> and <end_of_turn>
}

MODEL_REGISTRY: Dict[str, Dict[str, Any]] = {
    "gemma3": {
        "name": "Gemma 3 4B",
        "hf_id": "google/gemma-3-4b-it",
        "params_b": 4.0,
    },
    "mistral7b": {
        "name": "Mistral 7B",
        "hf_id": "mistralai/Mistral-7B-Instruct-v0.3",
        "params_b": 7.0,
    },
    "llama31": {
        "name": "Llama 3.1 8B",
        "hf_id": "meta-llama/Meta-Llama-3.1-8B-Instruct",
        "params_b": 8.0,
    },
    "qwen25": {
        "name": "Qwen 2.5 3B",
        "hf_id": "Qwen/Qwen2.5-3B-Instruct",
        "params_b": 3.0,
    },
    "smollm": {
        "name": "SmolLM 1.7B",
        "hf_id": "HuggingFaceTB/SmolLM2-1.7B-Instruct",
        "params_b": 1.7,
    },
}

TIERS: Dict[str, Dict[str, Any]] = {
    "fp16": {"label": "Tier 0 — FP16", "load_in_8bit": False, "load_in_4bit": False},
    "int8": {"label": "Tier 1 — INT8", "load_in_8bit": True,  "load_in_4bit": False},
    "int4": {"label": "Tier 2 — INT4", "load_in_8bit": False, "load_in_4bit": True},
}

# Domain configs
DOMAIN_REGISTRY: Dict[str, Dict[str, Any]] = {
    "general_nlp": {
        "name": "General NLP",
        "task": "sentiment",
        "dataset_id": "stanfordnlp/sst2",
        "dataset_split": "validation",
        "text_field": "sentence",
        "label_field": "label",
        "label_map": {0: "negative", 1: "positive"},
        "prompt_template": (
            "Classify the sentiment of the following text.\n"
            "Reply with exactly one word: 'positive' or 'negative'.\n"
            "Text: {text}\nAnswer:"
        ),
        "valid_answers": ["positive", "negative"],
        "metric": "accuracy",
    },
    "financial_nlp": {
        "name": "Financial NLP",
        "task": "sentiment",
        "dataset_id": "zeroshot/twitter-financial-news-sentiment",
        "dataset_split": "train",
        "text_field": "text",
        "label_field": "label",
        # 0=bearish(negative), 1=bullish(positive), 2=neutral
        "label_map": {0: "negative", 1: "positive", 2: "neutral"},
        "prompt_template": (
            "Classify the sentiment of this financial news as exactly "
            "'positive', 'negative', or 'neutral'.\n"
            "Text: {text}\nAnswer:"
        ),
        "valid_answers": ["positive", "negative", "neutral"],
        "metric": "accuracy",
    },
    "structured_reasoning": {
        "name": "Structured Reasoning",
        "task": "math",
        "dataset_id": "ChilleD/SVAMP",
        "dataset_split": "test",
        "text_field": "question_concat",   # Body + Question combined
        "label_field": "Answer",
        "label_map": {},
        "prompt_template": (
            "Solve this math problem step by step, then write the final answer "
            "on the last line as: Answer: <number>\n"
            "Problem: {text}"
        ),
        "valid_answers": [],   # any numeric string
        "metric": "exact_match",
    },
}

# ---------------------------------------------------------------------------
# Result dataclass
# ---------------------------------------------------------------------------
@dataclass
class CellResult:
    model_key: str
    model_name: str
    tier: str
    domain: str
    task: str
    n_samples: int
    accuracy: float
    f1_macro: float
    ece_proxy: float
    latency_ms_mean: float
    latency_ms_std: float
    status: str          # "ok" | "error"
    error_msg: str = ""
    timestamp: str = ""


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------
def gb(b: int) -> float:
    return round(b / (1024 ** 3), 2)


def flush_gpu(torch_mod) -> None:
    gc.collect()
    torch_mod.cuda.empty_cache()
    torch_mod.cuda.reset_peak_memory_stats()


def cell_id(model_key: str, tier: str, domain: str) -> str:
    return f"{model_key}__{tier}__{domain}"


def load_checkpoint(path: str) -> set:
    if os.path.exists(path):
        with open(path) as f:
            data = json.load(f)
        completed = set(data.get("completed", []))
        log.info(f"Checkpoint loaded — {len(completed)} cells already done.")
        return completed
    return set()


def save_checkpoint(path: str, completed: set) -> None:
    with open(path, "w") as f:
        json.dump({"completed": list(completed)}, f)


def append_result_csv(result: CellResult, path: str) -> None:
    """Append one result row to CSV. Creates header if file doesn't exist."""
    file_exists = os.path.exists(path)
    with open(path, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(asdict(result).keys()))
        if not file_exists:
            writer.writeheader()
        writer.writerow(asdict(result))


# ---------------------------------------------------------------------------
# Dependency imports
# ---------------------------------------------------------------------------
def import_dependencies() -> Dict[str, Any]:
    missing = []
    deps: Dict[str, Any] = {}

    try:
        import torch
        deps["torch"] = torch
    except ImportError:
        missing.append("torch")

    try:
        from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
        deps["AutoTokenizer"]        = AutoTokenizer
        deps["AutoModelForCausalLM"] = AutoModelForCausalLM
        deps["BitsAndBytesConfig"]   = BitsAndBytesConfig
    except ImportError:
        missing.append("transformers")

    try:
        from datasets import load_dataset
        deps["load_dataset"] = load_dataset
    except ImportError:
        missing.append("datasets")

    try:
        import numpy as np
        deps["np"] = np
    except ImportError:
        missing.append("numpy")

    try:
        from sklearn.metrics import f1_score
        deps["f1_score"] = f1_score
    except ImportError:
        missing.append("scikit-learn")

    if missing:
        raise ImportError(
            f"\n\nMissing: {' '.join(missing)}\n"
            f"Run: pip install {' '.join(missing)} accelerate bitsandbytes\n"
        )
    return deps


# ---------------------------------------------------------------------------
# Model loading
# ---------------------------------------------------------------------------
def load_model_and_tokenizer(
    model_key: str,
    tier_key: str,
    deps: Dict[str, Any],
    hf_token: Optional[str] = None,
) -> Tuple[Any, Any]:
    """Load model + tokenizer for a given model × tier. Returns (model, tokenizer)."""

    torch     = deps["torch"]
    AutoTok   = deps["AutoTokenizer"]
    AutoModel = deps["AutoModelForCausalLM"]
    BnBConfig = deps["BitsAndBytesConfig"]

    model_cfg = MODEL_REGISTRY[model_key]
    tier_cfg  = TIERS[tier_key]

    log.info(f"Loading {model_cfg['name']} @ {tier_cfg['label']} ...")
    flush_gpu(torch)

    tok_kwargs: Dict[str, Any] = {"trust_remote_code": True}
    if hf_token:
        tok_kwargs["token"] = hf_token

    tokenizer = AutoTok.from_pretrained(model_cfg["hf_id"], **tok_kwargs)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Quantization config
    bnb_cfg = None
    if tier_cfg["load_in_8bit"]:
        bnb_cfg = BnBConfig(load_in_8bit=True)
    elif tier_cfg["load_in_4bit"]:
        bnb_cfg = BnBConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
        )

    # Select dtype: bfloat16 for models that need it, float16 otherwise
    dtype = torch.bfloat16 if model_key in BFLOAT16_MODELS else torch.float16

    load_kwargs: Dict[str, Any] = {
        "device_map": "auto",
        "trust_remote_code": True,
    }
    if bnb_cfg is not None:
        load_kwargs["quantization_config"] = bnb_cfg
        # bitsandbytes quantized models still need compute dtype hint
        if model_key in BFLOAT16_MODELS:
            load_kwargs["torch_dtype"] = torch.bfloat16
    else:
        load_kwargs["dtype"] = dtype

    if model_key in EAGER_ATTN_MODELS:
        load_kwargs["attn_implementation"] = "eager"

    if hf_token:
        load_kwargs["token"] = hf_token

    model = AutoModel.from_pretrained(model_cfg["hf_id"], **load_kwargs)
    model.eval()

    vram_used = gb(torch.cuda.memory_allocated())
    log.info(f"  Loaded. VRAM: {vram_used} GB | dtype: {dtype}")
    return model, tokenizer


# ---------------------------------------------------------------------------
# Inference
# ---------------------------------------------------------------------------
def normalize_number(s: str) -> str:
    """
    Normalize a numeric string for comparison.
    '27.0' -> '27', '27' -> '27', '4.50' -> '4.5'
    """
    try:
        f = float(s)
        # If it's a whole number, return as int string
        if f == int(f):
            return str(int(f))
        return str(round(f, 4))
    except (ValueError, TypeError):
        return s.strip()


def extract_answer(output_text: str, valid_answers: List[str], task: str) -> str:
    """
    Extract the model's answer from generated text.
    For classification: match first valid answer token.
    For math: extract first number, normalize to int if whole number.
    """
    text = output_text.strip().lower()

    if task == "math":
        # Try to find "Answer: <number>" pattern first (chain-of-thought)
        cot_match = re.search(r"answer[:\s]+(-?\d+(?:\.\d+)?)", text)
        if cot_match:
            return normalize_number(cot_match.group(1))
        # Fallback: extract last number in output
        # (last number more likely to be the answer than the first)
        nums = re.findall(r"-?\d+(?:\.\d+)?", text)
        if not nums:
            return ""
        return normalize_number(nums[-1])

    # Classification: find first occurrence of any valid answer
    for ans in valid_answers:
        if ans.lower() in text:
            return ans.lower()
    # Fallback: return first word
    return text.split()[0] if text.split() else ""


def run_inference(
    model,
    tokenizer,
    prompt: str,
    valid_answers: List[str],
    task: str,
    deps: Dict[str, Any],
    model_key: str = "",
    max_new_tokens: int = 16,
) -> Tuple[str, float, float]:
    """
    Run one inference pass.
    Returns (predicted_answer, confidence_proxy, latency_ms).
    """
    torch = deps["torch"]

    # Use chat template for instruct models
    if model_key in CHAT_TEMPLATE_MODELS and hasattr(tokenizer, "apply_chat_template"):
        text = tokenizer.apply_chat_template(
            [{"role": "user", "content": prompt}],
            tokenize=False,
            add_generation_prompt=True,
        )
    else:
        text = prompt

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512,
        padding=False,
    ).to("cuda")

    # Build eos token list
    eos_ids = [tokenizer.eos_token_id] if tokenizer.eos_token_id is not None else []
    extra   = EXTRA_EOS_TOKENS.get(model_key, [])
    eos_ids = list(set(eos_ids + extra)) or None

    t0 = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,        # greedy for reproducibility
            pad_token_id=tokenizer.pad_token_id or 0,
            eos_token_id=eos_ids,
        )
    latency_ms = (time.time() - t0) * 1000

    # Decode only the newly generated tokens
    input_len = inputs["input_ids"].shape[1]
    generated = outputs[0][input_len:]
    decoded   = tokenizer.decode(generated, skip_special_tokens=True).strip()

    pred = extract_answer(decoded, valid_answers, task)

    # Confidence proxy: logit margin for classification,
    # 1.0 for math (no reliable proxy without beam scores)
    if task != "math" and valid_answers:
        try:
            with torch.no_grad():
                logits = model(
                    input_ids=inputs["input_ids"],
                    attention_mask=inputs["attention_mask"],
                ).logits[:, -1, :]
            # Check for NaN (can happen with fp16 on some models)
            if logits.isnan().any():
                confidence = 1.0 / max(len(valid_answers), 1)
            else:
                probs = torch.softmax(logits, dim=-1)
                ans_probs = []
                for ans in valid_answers:
                    tok_id = tokenizer.encode(f" {ans}", add_special_tokens=False)
                    if tok_id:
                        ans_probs.append(probs[0, tok_id[0]].item())
                    else:
                        ans_probs.append(0.0)
                denom = sum(ans_probs) + 1e-9
                confidence = max(ans_probs) / denom
        except Exception:
            confidence = 1.0 / max(len(valid_answers), 1)
    else:
        confidence = 1.0

    return pred, confidence, latency_ms


# ---------------------------------------------------------------------------
# Cell evaluation
# ---------------------------------------------------------------------------
def evaluate_cell(
    model_key: str,
    tier_key: str,
    domain_key: str,
    model,
    tokenizer,
    deps: Dict[str, Any],
    n_samples: int = SAMPLES_PER_CELL,
) -> CellResult:
    """
    Evaluate one model × tier × domain cell on clean data.
    Returns a CellResult with accuracy, F1, ECE, latency.
    """
    np       = deps["np"]
    f1_score = deps["f1_score"]
    load_dataset = deps["load_dataset"]

    domain_cfg = DOMAIN_REGISTRY[domain_key]
    model_cfg  = MODEL_REGISTRY[model_key]
    task       = domain_cfg["task"]

    log.info(
        f"  Evaluating {model_cfg['name']} | {TIERS[tier_key]['label']} | "
        f"{domain_cfg['name']} | {n_samples} samples ..."
    )

    try:
        # Load dataset (streaming to avoid full download)
        ds_kwargs: Dict[str, Any] = {
            "split": domain_cfg["dataset_split"],
            "streaming": True,
            "trust_remote_code": True,
        }
        ds = load_dataset(domain_cfg["dataset_id"], **ds_kwargs)
        samples = list(ds.take(n_samples))

        preds, labels, confs, lats = [], [], [], []

        for i, sample in enumerate(samples):
            text  = str(sample[domain_cfg["text_field"]])
            label = sample[domain_cfg["label_field"]]

            # Map numeric label to string for classification
            if domain_cfg["label_map"] and isinstance(label, int):
                label_str = domain_cfg["label_map"].get(label, str(label))
            elif task == "math":
                # Normalize numeric labels e.g. '27' or 27.0 -> '27'
                label_str = normalize_number(str(label))
            else:
                label_str = str(label).strip().lower()

            prompt = domain_cfg["prompt_template"].format(text=text[:400])

            pred, conf, lat = run_inference(
                model, tokenizer, prompt,
                domain_cfg["valid_answers"], task, deps,
                model_key=model_key,
                max_new_tokens=150 if task == "math" else 16,
            )

            preds.append(pred)
            labels.append(label_str)
            confs.append(conf)
            lats.append(lat)

            if (i + 1) % 10 == 0:
                running_acc = sum(p == l for p, l in zip(preds, labels)) / len(preds)
                log.info(f"    [{i+1}/{n_samples}] running acc={running_acc:.3f}")

        # Metrics
        correct = [p == l for p, l in zip(preds, labels)]
        accuracy = float(np.mean(correct))

        # F1 (macro for multi-class, binary for binary)
        unique_labels = list(set(labels))
        if len(unique_labels) > 1:
            try:
                f1 = float(f1_score(labels, preds, average="macro", zero_division=0))
            except Exception:
                f1 = accuracy
        else:
            f1 = accuracy

        # ECE proxy: mean |confidence - correctness|
        ece = float(np.mean([abs(c - int(ok)) for c, ok in zip(confs, correct)]))

        lat_arr = np.array(lats)
        lat_mean = float(np.mean(lat_arr))
        lat_std  = float(np.std(lat_arr))

        log.info(
            f"  Done — acc={accuracy:.3f} f1={f1:.3f} "
            f"ece={ece:.3f} lat={lat_mean:.1f}±{lat_std:.1f}ms"
        )

        return CellResult(
            model_key=model_key,
            model_name=model_cfg["name"],
            tier=tier_key,
            domain=domain_key,
            task=task,
            n_samples=len(samples),
            accuracy=round(accuracy, 4),
            f1_macro=round(f1, 4),
            ece_proxy=round(ece, 4),
            latency_ms_mean=round(lat_mean, 2),
            latency_ms_std=round(lat_std, 2),
            status="ok",
            timestamp=time.strftime("%Y-%m-%dT%H:%M:%S"),
        )

    except Exception as exc:
        log.error(f"  Cell failed: {exc}")
        log.debug(traceback.format_exc())
        return CellResult(
            model_key=model_key,
            model_name=model_cfg["name"],
            tier=tier_key,
            domain=domain_key,
            task=task,
            n_samples=0,
            accuracy=0.0,
            f1_macro=0.0,
            ece_proxy=0.0,
            latency_ms_mean=0.0,
            latency_ms_std=0.0,
            status="error",
            error_msg=str(exc)[:200],
            timestamp=time.strftime("%Y-%m-%dT%H:%M:%S"),
        )


# ---------------------------------------------------------------------------
# Main orchestration
# ---------------------------------------------------------------------------
def parse_args() -> argparse.Namespace:
    p = argparse.ArgumentParser(
        description="Phase 1 baseline evaluation — Fragility Under Constraint"
    )
    p.add_argument(
        "--model",
        choices=list(MODEL_REGISTRY.keys()) + ["all"],
        default="all",
        help="Which model(s) to run. Default: all.",
    )
    p.add_argument(
        "--tiers",
        nargs="+",
        choices=list(TIERS.keys()),
        default=["fp16", "int8", "int4"],
        help="Which tiers to evaluate. Default: all three.",
    )
    p.add_argument(
        "--domains",
        nargs="+",
        choices=list(DOMAIN_REGISTRY.keys()),
        default=list(DOMAIN_REGISTRY.keys()),
        help="Which domains to evaluate. Default: all three.",
    )
    p.add_argument(
        "--samples",
        type=int,
        default=SAMPLES_PER_CELL,
        help=f"Samples per cell. Default: {SAMPLES_PER_CELL}.",
    )
    p.add_argument(
        "--resume",
        action="store_true",
        help="Resume from checkpoint — skip already-completed cells.",
    )
    p.add_argument(
        "--hf-token",
        type=str,
        default=os.environ.get("HF_TOKEN"),
        help="HuggingFace token. Can also set HF_TOKEN env variable.",
    )
    p.add_argument(
        "--output-dir",
        type=str,
        default="/kaggle/working",
        help="Directory to write results and checkpoints.",
    )
    return p.parse_args()


def main() -> None:
    args = parse_args()

    log.info("=" * 60)
    log.info("Fragility Under Constraint — Phase 1: Baselines")
    log.info("=" * 60)

    # Setup output paths
    out_dir = Path(args.output_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_path = str(out_dir / CHECKPOINT_FILE)
    csv_path        = str(out_dir / RESULTS_CSV)
    json_path       = str(out_dir / RESULTS_JSON)

    # Imports
    try:
        deps = import_dependencies()
    except ImportError as e:
        log.error(str(e))
        sys.exit(1)

    torch = deps["torch"]

    if not torch.cuda.is_available():
        log.error("CUDA not available. Run on a GPU instance.")
        sys.exit(1)

    log.info(f"GPU: {torch.cuda.get_device_name(0)}")
    log.info(f"VRAM: {gb(torch.cuda.get_device_properties(0).total_memory)} GB")

    # Build experiment plan
    model_keys  = list(MODEL_REGISTRY.keys()) if args.model == "all" else [args.model]
    tier_keys   = args.tiers
    domain_keys = args.domains

    total_cells = len(model_keys) * len(tier_keys) * len(domain_keys)
    log.info(
        f"\nPlan: {len(model_keys)} models × {len(tier_keys)} tiers × "
        f"{len(domain_keys)} domains = {total_cells} cells"
    )
    log.info(f"Samples per cell: {args.samples}")

    # Load checkpoint if resuming
    completed = load_checkpoint(checkpoint_path) if args.resume else set()

    # Track all results for final JSON
    all_results: List[CellResult] = []
    cell_num = 0

    # Outer loop: model × tier (load model once per model+tier combo)
    for model_key in model_keys:
        for tier_key in tier_keys:

            # Check if all domains for this model+tier are already done
            pending_domains = [
                d for d in domain_keys
                if cell_id(model_key, tier_key, d) not in completed
            ]
            if not pending_domains:
                log.info(
                    f"Skipping {MODEL_REGISTRY[model_key]['name']} @ "
                    f"{TIERS[tier_key]['label']} — all domains done."
                )
                continue

            # Load model once for all domains in this tier
            try:
                model, tokenizer = load_model_and_tokenizer(
                    model_key, tier_key, deps, hf_token=args.hf_token
                )
            except Exception as exc:
                log.error(
                    f"Failed to load {MODEL_REGISTRY[model_key]['name']} @ "
                    f"{TIERS[tier_key]['label']}: {exc}"
                )
                # Log error result for all pending domains
                for domain_key in pending_domains:
                    cell_num += 1
                    result = CellResult(
                        model_key=model_key,
                        model_name=MODEL_REGISTRY[model_key]["name"],
                        tier=tier_key,
                        domain=domain_key,
                        task=DOMAIN_REGISTRY[domain_key]["task"],
                        n_samples=0,
                        accuracy=0.0, f1_macro=0.0, ece_proxy=0.0,
                        latency_ms_mean=0.0, latency_ms_std=0.0,
                        status="error",
                        error_msg=f"Model load failed: {str(exc)[:150]}",
                        timestamp=time.strftime("%Y-%m-%dT%H:%M:%S"),
                    )
                    all_results.append(result)
                    append_result_csv(result, csv_path)
                continue

            # Inner loop: domains (model already loaded)
            for domain_key in pending_domains:
                cell_num += 1
                cid = cell_id(model_key, tier_key, domain_key)
                log.info(
                    f"\n[Cell {cell_num}/{total_cells}] "
                    f"{MODEL_REGISTRY[model_key]['name']} | "
                    f"{TIERS[tier_key]['label']} | "
                    f"{DOMAIN_REGISTRY[domain_key]['name']}"
                )

                result = evaluate_cell(
                    model_key, tier_key, domain_key,
                    model, tokenizer, deps,
                    n_samples=args.samples,
                )
                all_results.append(result)

                # Write to CSV immediately (safe against crashes)
                append_result_csv(result, csv_path)

                # Update checkpoint
                completed.add(cid)
                save_checkpoint(checkpoint_path, completed)

                log.info(
                    f"  ✓ Cell saved. "
                    f"acc={result.accuracy:.3f} f1={result.f1_macro:.3f} "
                    f"ece={result.ece_proxy:.3f} lat={result.latency_ms_mean:.1f}ms"
                )

            # Unload model before loading next one
            log.info(
                f"Unloading {MODEL_REGISTRY[model_key]['name']} @ "
                f"{TIERS[tier_key]['label']} ..."
            )
            del model, tokenizer
            flush_gpu(torch)

    # Write full JSON results
    with open(json_path, "w") as f:
        json.dump([asdict(r) for r in all_results], f, indent=2)

    # Summary table
    log.info("\n" + "=" * 60)
    log.info("PHASE 1 COMPLETE — BASELINE RESULTS SUMMARY")
    log.info("=" * 60)
    log.info(
        f"{'Model':20s} {'Tier':6s} {'Domain':22s} "
        f"{'Acc':6s} {'F1':6s} {'ECE':6s} {'Lat(ms)':8s} {'Status':6s}"
    )
    log.info("-" * 80)
    for r in all_results:
        log.info(
            f"{r.model_name:20s} {r.tier:6s} {r.domain:22s} "
            f"{r.accuracy:6.3f} {r.f1_macro:6.3f} {r.ece_proxy:6.3f} "
            f"{r.latency_ms_mean:8.1f} {r.status:6s}"
        )

    ok_count  = sum(1 for r in all_results if r.status == "ok")
    err_count = sum(1 for r in all_results if r.status == "error")
    log.info(f"\nCells completed: {ok_count} ok, {err_count} errors")
    log.info(f"Results saved to: {csv_path}")
    log.info(f"Full JSON saved to: {json_path}")
    log.info("=" * 60)


if __name__ == "__main__":
    main()

Writing phase1_baselines.py


In [5]:
# Clear everything
import os
for f in ["baseline_results.csv", "phase1_checkpoint.json"]:
    path = f"/kaggle/working/{f}"
    if os.path.exists(path): os.remove(path)
print("Cleared.")

Cleared.


In [6]:
# Full 45-cell run
!python phase1_baselines.py \
    --hf-token {os.environ["HF_TOKEN"]} \
    --output-dir /kaggle/working

06:37:15  INFO      ============================================================
06:37:15  INFO      Fragility Under Constraint — Phase 1: Baselines
06:37:15  INFO      ============================================================
06:37:33  INFO      NumExpr defaulting to 4 threads.
06:37:35  INFO      TensorFlow version 2.19.0 available.
06:37:35  INFO      JAX version 0.7.2 available.
06:37:36  INFO      GPU: Tesla T4
06:37:36  INFO      VRAM: 14.56 GB
06:37:36  INFO      
Plan: 5 models × 3 tiers × 3 domains = 45 cells
06:37:36  INFO      Samples per cell: 50
06:37:36  INFO      Loading Gemma 3 4B @ Tier 0 — FP16 ...
06:37:36  INFO      HTTP Request: HEAD https://huggingface.co/google/gemma-3-4b-it/resolve/main/config.json "HTTP/1.1 200 OK"
06:37:36  INFO      HTTP Request: GET https://huggingface.co/google/gemma-3-4b-it/resolve/main/config.json "HTTP/1.1 200 OK"
config.json: 100%|█████████████████████████████| 855/855 [00:00<00:00, 4.41MB/s]
06:37:37  INFO      HTTP Request: HEAD ht